In [ ]:
import math
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

try:
    import gymnasium as gym
except ImportError as exc:
    raise ImportError("gymnasium is required. Install gymnasium[mujoco] before running this notebook.") from exc

from based_cnn_model import DiffusionPolicy


In [ ]:
@dataclass
class TrainConfig:
    env_id: str = "Ant-v5"
    seed: int = 42
    horizon: int = 16
    diffusion_steps: int = 50
    obs_latent: int = 128
    pos_latent: int = 64
    rollout_steps: int = 6000
    batch_size: int = 64
    num_epochs: int = 20
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    beta_start: float = 1e-4
    beta_end: float = 2e-2
    log_every: int = 50

config = TrainConfig()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

device


In [ ]:
data = np.load("./datas/reach_bc.npz")
data

In [ ]:
env = gym.make(config.env_id)
obs_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
action_low = env.action_space.low.astype(np.float32)
action_high = env.action_space.high.astype(np.float32)

print(f"obs_dim={obs_dim}, action_dim={action_dim}")


In [ ]:
def handcrafted_teacher(obs: np.ndarray, horizon: int, action_dim: int) -> np.ndarray:
    """
    실제 expert policy가 없을 때 action chunk 형식을 맞추기 위한 임시 teacher.
    obs의 일부 차원을 섞어서 horizon 길이의 action sequence를 만든다.
    반환 shape: (horizon, action_dim)
    """
    base = np.tanh(obs[:action_dim])
    drift_src = obs[action_dim: action_dim * 2]
    if drift_src.shape[0] < action_dim:
        drift_src = np.pad(drift_src, (0, action_dim - drift_src.shape[0]))
    drift = 0.15 * np.tanh(drift_src[:action_dim])

    chunk = []
    for step in range(horizon):
        phase = step / max(horizon - 1, 1)
        action = base + phase * drift
        action += 0.03 * np.sin(phase * np.pi * (np.arange(action_dim) + 1))
        chunk.append(np.clip(action, -1.0, 1.0).astype(np.float32))
    return np.stack(chunk, axis=0)


def collect_example_dataset(env, rollout_steps: int, horizon: int, seed: int):
    observations = []
    action_chunks = []

    obs, _ = env.reset(seed=seed)
    for _ in range(rollout_steps):
        observations.append(obs.astype(np.float32))
        action_chunks.append(handcrafted_teacher(obs, horizon, action_dim))

        exploratory_action = env.action_space.sample()
        next_obs, _, terminated, truncated, _ = env.step(exploratory_action)
        done = terminated or truncated
        obs = next_obs
        if done:
            obs, _ = env.reset()

    observations = np.asarray(observations, dtype=np.float32)
    action_chunks = np.asarray(action_chunks, dtype=np.float32)
    return observations, action_chunks


In [ ]:
observations, action_chunks = collect_example_dataset(
    env=env,
    rollout_steps=config.rollout_steps,
    horizon=config.horizon,
    seed=config.seed,
)

print(observations.shape)
print(action_chunks.shape)


In [ ]:
class AntDiffusionDataset(Dataset):
    def __init__(self, observations: np.ndarray, action_chunks: np.ndarray):
        self.observations = torch.from_numpy(observations)
        self.action_chunks = torch.from_numpy(action_chunks)

    def __len__(self):
        return len(self.observations)

    def __getitem__(self, idx):
        obs = self.observations[idx]
        action_chunk = self.action_chunks[idx]
        action_chunk = action_chunk.transpose(0, 1)
        return {"obs": obs, "action_chunk": action_chunk}


dataset = AntDiffusionDataset(observations, action_chunks)
dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True, drop_last=True)

batch = next(iter(dataloader))
print(batch["obs"].shape, batch["action_chunk"].shape)


In [ ]:
model = DiffusionPolicy(
    action_dim=action_dim,
    obs_dim=obs_dim,
    obs_latent=config.obs_latent,
    pos_latent=config.pos_latent,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)

betas, alphas, alpha_bars = model.get_b_a_ab(
    start=config.beta_start,
    end=config.beta_end,
    timestep=config.diffusion_steps,
)
alpha_bars = alpha_bars.to(device)

sum(p.numel() for p in model.parameters())


In [ ]:
def q_sample_actions(x0: torch.Tensor, t_index: torch.Tensor, alpha_bars: torch.Tensor):
    noise = torch.randn_like(x0)
    alpha_bar_t = alpha_bars[t_index].view(-1, 1, 1)
    xt = torch.sqrt(alpha_bar_t) * x0 + torch.sqrt(1.0 - alpha_bar_t) * noise
    return xt, noise


def train_one_epoch(model, dataloader, optimizer, alpha_bars, diffusion_steps, device, log_every):
    model.train()
    running_loss = 0.0

    for step, batch in enumerate(dataloader, start=1):
        obs = batch["obs"].to(device)
        action_chunk = batch["action_chunk"].to(device)

        t_index = torch.randint(0, diffusion_steps, (obs.shape[0],), device=device)
        noisy_action, noise = q_sample_actions(action_chunk, t_index, alpha_bars)

        pred_noise = model(noisy_action, t_index.float(), obs)
        loss = F.mse_loss(pred_noise, noise)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        if step % log_every == 0:
            print(f"step={step:04d} loss={running_loss / step:.6f}")

    return running_loss / max(len(dataloader), 1)


In [ ]:
history = []

for epoch in range(1, config.num_epochs + 1):
    epoch_loss = train_one_epoch(
        model=model,
        dataloader=dataloader,
        optimizer=optimizer,
        alpha_bars=alpha_bars,
        diffusion_steps=config.diffusion_steps,
        device=device,
        log_every=config.log_every,
    )
    history.append(epoch_loss)
    print(f"epoch={epoch:03d} mean_loss={epoch_loss:.6f}")


In [ ]:
@torch.no_grad()
def sample_action_chunk(model, obs: torch.Tensor, horizon: int, action_dim: int, alpha_bars: torch.Tensor, beta_start: float, beta_end: float, diffusion_steps: int):
    model.eval()
    x = torch.randn(1, action_dim, horizon, device=obs.device)

    betas, alphas, _ = model.get_b_a_ab(beta_start, beta_end, diffusion_steps)
    betas = betas.to(obs.device)
    alphas = alphas.to(obs.device)

    for step in reversed(range(diffusion_steps)):
        t_batch = torch.full((1,), float(step), device=obs.device)
        pred_noise = model(x, t_batch, obs)

        alpha_t = alphas[step]
        alpha_bar_t = alpha_bars[step]
        coef = (1.0 - alpha_t) / torch.sqrt(1.0 - alpha_bar_t)
        x = (x - coef * pred_noise) / torch.sqrt(alpha_t)

        if step > 0:
            x = x + torch.sqrt(betas[step]) * torch.randn_like(x)

    return x.squeeze(0).transpose(0, 1)


test_obs = torch.from_numpy(observations[0]).unsqueeze(0).to(device)
sampled_chunk = sample_action_chunk(
    model=model,
    obs=test_obs,
    horizon=config.horizon,
    action_dim=action_dim,
    alpha_bars=alpha_bars,
    beta_start=config.beta_start,
    beta_end=config.beta_end,
    diffusion_steps=config.diffusion_steps,
)
sampled_chunk.shape
